<a href="https://colab.research.google.com/github/tardigrade-dot/colab-script/blob/main/train/gomoku.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install datasets safetensors torch

In [ ]:
"""
Gomoku AlphaZero-style Training — Colab Edition v2
===================================================
修复列表（相对上一版本）：
  1. board_size 18 → 20  （根据探测结果修正为20x20）
  2. 数据过滤：
       - 跳过 <20步 或 >70步 的低质量对局
       - 只取每局后60%的步数（前期布局信息量低）
       - 跳过棋盘上棋子数<10的局面
  3. batch_size 512 → 1024，充分利用A100显存
  4. lr 5e-4 → 7e-4（batch翻倍后同步调整）
  5. warmup scheduler：前500步线性升温，避免早期震荡
  6. epochs 10 → 5
  7. num_workers 2 → 4
  8. 训练日志加入 policy/value 分项显示
"""

import os, time, random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler

# ── 1. Config ──────────────────────────────────────────────────────────────────

@dataclass
class Config:
    # Board — 数据集探测结果为 20x20 (max_pos=399)
    board_size: int = 20
    n_cells: int = field(init=False)

    # Model
    channels: int = 128
    n_blocks: int = 7
    policy_channels: int = 4
    value_channels: int = 8
    value_hidden: int = 256

    # Data filtering
    min_moves: int = 20
    max_moves: int = 70
    min_stones: int = 10
    late_ratio: float = 0.4

    # Training
    epochs: int = 5
    batch_size: int = 1024 * 4
    lr: float = 7e-4
    warmup_steps: int = 500
    lr_milestones: list = field(default_factory=lambda: [2, 4])
    lr_gamma: float = 0.3
    weight_decay: float = 1e-4
    val_ratio: float = 0.02
    max_games: Optional[int] = None

    # Loss weights
    policy_weight: float = 1.0
    value_weight: float = 1.0

    # Misc
    num_workers: int = 8
    seed: int = 42
    save_path: str = "/content/drive/MyDrive/gomoku_model.safetensors"
    checkpoint_every: int = 1

    def __post_init__(self):
        self.n_cells = self.board_size * self.board_size


CFG = Config()
torch.manual_seed(CFG.seed)
random.seed(CFG.seed)
np.random.seed(CFG.seed)

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── 2. Win Detection ───────────────────────────────────────────────────────────

DIRS = [(0, 1), (1, 0), (1, 1), (1, -1)]

def check_win(board: np.ndarray, r: int, c: int, player: int, size: int) -> bool:
    for dr, dc in DIRS:
        count = 1
        for sign in (1, -1):
            nr, nc = r + sign * dr, c + sign * dc
            while 0 <= nr < size and 0 <= nc < size and board[nr, nc] == player:
                count += 1
                nr += sign * dr
                nc += sign * dc
        if count >= 5:
            return True
    return False


def replay_game(moves: list, size: int) -> Optional[int]:
    board = np.zeros((size, size), dtype=np.int8)
    for i, pos in enumerate(moves):
        r, c = divmod(pos, size)
        if r >= size or c >= size:
            return None
        player = i % 2
        board[r, c] = player + 1
        if check_win(board, r, c, player + 1, size):
            return player
    return None

# ── 3. Dataset ─────────────────────────────────────────────────────────────────

def build_state(moves: list, step: int, size: int) -> np.ndarray:
    state = np.zeros((2, size, size), dtype=np.float32)
    current_player = step % 2
    for i in range(step):
        r, c = divmod(moves[i], size)
        ch = 0 if (i % 2) == current_player else 1
        state[ch, r, c] = 1.0
    return state


class GomokuDataset(Dataset):
    def __init__(self, hf_ds, metadata: list, cfg: Config):
        self.hf_ds = hf_ds
        self.metadata = metadata
        self.cfg = cfg

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row_idx, step, winner = self.metadata[idx]
        moves = self.hf_ds[row_idx]["input_ids"]
        size = self.cfg.board_size

        state = build_state(moves, step, size)

        policy = np.zeros(self.cfg.n_cells, dtype=np.float32)
        policy[moves[step]] = 1.0

        value = 1.0 if (step % 2) == winner else 0.0

        return (
            torch.from_numpy(state),
            torch.from_numpy(policy),
            torch.tensor(value, dtype=torch.float32),
        )


def prepare_metadata(cfg: Config):
    from datasets import load_dataset
    try:
        from tqdm import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    print("Loading dataset …")
    hf_ds = load_dataset("PoolC/gomoku-dataset-1.8M-fixed", split="train")

    metadata = []
    stats = {"total": 0, "short": 0, "long": 0, "no_winner": 0, "ok": 0}
    size = cfg.board_size

    for i, row in enumerate(tqdm(hf_ds, desc="Scanning games")):
        moves = row["input_ids"]
        stats["total"] += 1

        if len(moves) < cfg.min_moves:
            stats["short"] += 1
            continue
        if len(moves) > cfg.max_moves:
            stats["long"] += 1
            continue

        winner = replay_game(moves, size)
        if winner is None:
            stats["no_winner"] += 1
            continue

        stats["ok"] += 1
        total_steps = len(moves)
        start_step = int(total_steps * cfg.late_ratio)

        for step in range(start_step, total_steps):
            if step < cfg.min_stones:
                continue
            metadata.append((i, step, winner))

        if cfg.max_games and stats["ok"] >= cfg.max_games:
            break

    print(f"\nGames — total: {stats['total']:,} ok: {stats['ok']:,} Training positions: {len(metadata):,}")
    return hf_ds, metadata

# ── 4. Model ───────────────────────────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.norm2 = nn.LayerNorm(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)

    def _norm_channel(self, x: torch.Tensor, norm: nn.LayerNorm) -> torch.Tensor:
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(-1, C)
        x = norm(x)
        return x.reshape(B, H, W, C).permute(0, 3, 1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self._norm_channel(x, self.norm1)
        out = F.silu(out)
        out = self.conv1(out)
        out = self._norm_channel(out, self.norm2)
        out = F.silu(out)
        out = self.conv2(out)
        return out + residual


class PolicyValueNet(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        C = cfg.channels
        sq = cfg.n_cells
        self.stem = nn.Conv2d(2, C, 3, padding=1, bias=True)
        self.blocks = nn.ModuleList([ResBlock(C) for _ in range(cfg.n_blocks)])
        pc = cfg.policy_channels
        self.policy_conv = nn.Conv2d(C, pc, 1, bias=True)
        self.policy_fc = nn.Linear(pc * sq, sq, bias=True)
        vc = cfg.value_channels
        self.value_conv = nn.Conv2d(C, vc, 1, bias=True)
        self.value_fc1 = nn.Linear(vc * sq, cfg.value_hidden, bias=True)
        self.value_fc2 = nn.Linear(cfg.value_hidden, 1, bias=True)

    def forward(self, x: torch.Tensor):
        x = F.silu(self.stem(x))
        for block in self.blocks:
            x = block(x)
        B = x.size(0)
        p = F.silu(self.policy_conv(x))
        p = p.reshape(B, -1)
        p = self.policy_fc(p)
        v = F.silu(self.value_conv(x))
        v = v.reshape(B, -1)
        v = F.silu(self.value_fc1(v))
        v = self.value_fc2(v)
        return p, v

# ── 5. Loss ────────────────────────────────────────────────────────────────────

def compute_loss(policy_logits, value_logits, policy_target, value_target, cfg):
    log_probs = F.log_softmax(policy_logits, dim=-1)
    policy_loss = -(policy_target * log_probs).sum(dim=-1).mean()
    value_loss = F.binary_cross_entropy_with_logits(value_logits.squeeze(-1), value_target)
    total = cfg.policy_weight * policy_loss + cfg.value_weight * value_loss
    return total, {"policy": policy_loss.item(), "value": value_loss.item()}

# ── 6. Scheduler ───────────────────────────────────────────────────────────────

def build_scheduler(optimizer, cfg, steps_per_epoch):
    warmup = cfg.warmup_steps
    def lr_lambda(current_step):
        if current_step < warmup: return current_step / max(1, warmup)
        return 1.0
    warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    step_milestones = [m * steps_per_epoch for m in cfg.lr_milestones]
    main_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=step_milestones, gamma=cfg.lr_gamma)
    return warmup_scheduler, main_scheduler, warmup

# ── 7. Train / Val ─────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scaler, schedulers, cfg, epoch, global_step):
    warmup_sched, main_sched, warmup_steps = schedulers
    model.train()
    total_loss = p_sum = v_sum = 0.0
    t0 = time.time()
    for step, (states, policies, values) in enumerate(loader):
        states, policies, values = states.to(DEVICE), policies.to(DEVICE), values.to(DEVICE)
        optimizer.zero_grad()
        with autocast(enabled=(DEVICE == "cuda")):
            p_log, v_log = model(states)
            loss, sub = compute_loss(p_log, v_log, policies, values, cfg)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if global_step < warmup_steps: warmup_sched.step()
        else: main_sched.step()
        global_step += 1
        total_loss += loss.item(); p_sum += sub["policy"]; v_sum += sub["value"]
        if step % 200 == 0:
            print(f"  Epoch {epoch} | step {step}/{len(loader)} | loss {loss.item():.4f} | lr {optimizer.param_groups[0]['lr']:.2e}")
    return total_loss / len(loader), p_sum / len(loader), v_sum / len(loader), global_step

@torch.no_grad()
def val_epoch(model, loader, cfg):
    model.eval()
    total_loss = p_sum = v_sum = 0.0
    for states, policies, values in loader:
        states, policies, values = states.to(DEVICE), policies.to(DEVICE), values.to(DEVICE)
        p_log, v_log = model(states)
        _, sub = compute_loss(p_log, v_log, policies, values, cfg)
        p_sum += sub["policy"]; v_sum += sub["value"]
        total_loss += cfg.policy_weight * sub["policy"] + cfg.value_weight * sub["value"]
    return total_loss / len(loader), p_sum / len(loader), v_sum / len(loader)

def save_safetensors(model, path):
    from safetensors.torch import save_file
    state = {k: v.cpu().float() for k, v in model.state_dict().items()}
    save_file(state, path)
    print(f"  Saved: {path}")

def main():
    cfg = CFG
    hf_ds, metadata = prepare_metadata(cfg)
    full_ds = GomokuDataset(hf_ds, metadata, cfg)
    n_val = int(len(full_ds) * cfg.val_ratio)
    train_ds, val_ds = random_split(full_ds, [len(full_ds)-n_val, n_val], generator=torch.Generator().manual_seed(cfg.seed))
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size*2, shuffle=False, num_workers=cfg.num_workers)
    model = PolicyValueNet(cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(DEVICE == "cuda"))
    schedulers = build_scheduler(optimizer, cfg, len(train_loader))
    best_val_loss = float("inf"); global_step = 0
    for epoch in range(1, cfg.epochs + 1):
        tr_l, tr_p, tr_v, global_step = train_epoch(model, train_loader, optimizer, scaler, schedulers, cfg, epoch, global_step)
        va_l, va_p, va_v = val_epoch(model, val_loader, cfg)
        print(f"\nEpoch {epoch}/{cfg.epochs} | Train loss={tr_l:.4f} | Val loss={va_l:.4f}")
        if va_l < best_val_loss:
            best_val_loss = va_l
            save_safetensors(model, cfg.save_path)

if __name__ == "__main__":
    main()

In [ ]:
# 在Colab里单独跑这个cell
from datasets import load_dataset
ds = load_dataset("PoolC/gomoku-dataset-1.8M-fixed", split="train")

max_pos = 0
for row in ds:
    if row["input_ids"]:
        max_pos = max(max_pos, max(row["input_ids"]))

print(f"全局 max_pos = {max_pos}")
print(f"对应棋盘尺寸: {max_pos+1} cells")
# 如果 max_pos=360 → 19×19
# 如果 max_pos=399 → 20×20
# 如果 max_pos=323 → 18×18 但有越界说明数据有脏数据

In [ ]:
from datasets import load_dataset
ds = load_dataset("PoolC/gomoku-dataset-1.8M-fixed", split="train[:100]")

# 看一下原始数据的最大值
for row in ds:
    moves = row["input_ids"]
    print(f"len={len(moves)}, max_pos={max(moves)}, min_pos={min(moves)}")
    break